<style>
.hero{padding:38px 42px;border-radius:24px;color:#f8fbff;background:linear-gradient(125deg,#102a43,#184e77 55%,#2a9d8f);box-shadow:0 16px 38px #102a433d;font-family:Inter,'Segoe UI',sans-serif}.hero h1{font-size:38px;margin:12px 0;color:white}.chips span{display:inline-block;padding:6px 12px;margin:3px;border:1px solid #ffffff55;border-radius:999px;background:#ffffff20;font-size:12px;font-weight:700}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:12px}.card{padding:16px;border-radius:12px;background:#ffffff16;border:1px solid #ffffff38}.callout{margin:18px 0;padding:20px 24px;border-left:6px solid #2a9d8f;border-radius:12px;background:#edf7f6;color:#172033}.warn{border-left-color:#e9a23b;background:#fff8e8}.section{margin-top:28px;padding:15px 22px;border-radius:13px;background:linear-gradient(90deg,#102a43,#184e77);color:white}.metric{display:inline-block;min-width:150px;padding:15px;margin:5px;border-radius:12px;background:#edf5fb;border:1px solid #c9d9e6;text-align:center}.metric b{display:block;font-size:24px;color:#184e77}table{width:100%}th{background:#184e77!important;color:white!important;text-align:left!important}td,th{padding:9px!important}code{background:#102a4310;padding:2px 5px;border-radius:4px}</style><div class="hero"><div class="chips"><span>DEEP LEARNING</span><span>PROYECTO 1</span><span>VERSIÓN 2</span><span>SECCIÓN 30</span></div><h1>Monitoreo transaccional y detección de fraude</h1><p style="font-size:20px">Variables causales, validación temporal, boosting, PCA y costo</p><div class="grid"><div class="card"><b>Universidad</b><br>Universidad del Valle de Guatemala</div><div class="card"><b>Curso</b><br>Deep Learning y Sistemas Inteligentes</div><div class="card"><b>Docente</b><br>Kevin Recinos</div><div class="card"><b>Integrantes</b><br>Wilson Calderón · 22018<br>Pablo Barillas · 22193</div><div class="card"><b>Grupo</b><br>Grupo 1 · Sección 30</div><div class="card"><b>Período</b><br>Semestre II · 2026</div></div></div><div class="callout"><b>Propósito.</b> Mejorar datos, protocolo y decisión antes de aumentar complejidad. V1 permanece congelada y los resultados negativos se conservan.</div>

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd().resolve()
while not (ROOT/'artefactos').exists() and ROOT != ROOT.parent:
    ROOT=ROOT.parent
R=json.loads((ROOT/'artefactos/v2/resultados_v2.json').read_text(encoding='utf-8'))
pd.DataFrame(R['datos']['particiones']).T

,n,fraude,dt_min,dt_max
audit_train,324797.0,0.033870,86400.0,8022306.0
train,413378.0,0.035169,86400.0,10437996.0
validation,88581.0,0.034341,10438003.0,13151840.0
benchmark_historico,88581.0,0.034804,13151880.0,15811131.0


<div class="section"><h2>1 · Pregunta y protocolo</h2></div>

La pregunta es si el historial aporta señal incremental frente a una representación tabular fuerte. La V1 mostró una caída de AUC-PR de solo 0.0017 al permutar el orden; por eso V2 prioriza variables, identidad y validación. El costo es $$C(\tau)=4200FN(\tau)+180FP(\tau).$$ El último 15% es benchmark histórico reutilizado, no test ciego.

In [2]:
pd.DataFrame(R['identidad_secuencial']).T

,columnas,entidades,mediana_transacciones,p90_transacciones,porcentaje_con_3,porcentaje_con_8,porcentaje_con_16,porcentaje_con_32
tarjeta_direccion,"[card1, card2, card3, card5, addr1]",42946,2.0,21.0,94.898059,87.728012,80.228774,70.39303
tarjeta_direccion_correo,"[card1, card2, addr1, P_emaildomain]",92690,2.0,11.0,87.364785,73.036035,60.537813,47.692451
tarjeta_dispositivo_producto,"[card1, DeviceInfo, DeviceType, ProductCD]",44308,1.0,12.0,93.832933,87.892099,83.016392,77.033563
tarjeta_dispositivo,"[card1, DeviceInfo, DeviceType]",38600,2.0,14.0,94.819826,89.260169,84.569377,78.836489


<div class="section"><h2>2 · Datos y causalidad</h2></div>

`TransactionID` solo une; `TransactionDT` solo ordena. Tarjetas y direcciones son códigos, no magnitudes. Cada agregado satisface $$x_t^{hist}=f(\{x_j:t_j<t\}),$$ e incluye conteos previos, monto histórico, razón de monto, tiempo anterior y actividad 1/6/24/72 h.

In [3]:
pd.read_csv(ROOT/'datos/processed/v2/auditoria_variables.csv').sort_values('puntaje_relevancia',ascending=False).head(20)

,variable,tipo,ausencia,unicos,varianza,pearson,spearman,decision,informacion_mutua,puntaje_relevancia
275,V244,float64,0.736522,22,0.411668,0.353989,0.396162,candidata,0.023757,0.402101
273,V242,float64,0.736522,20,0.381891,0.349988,0.388830,candidata,0.022726,0.394511
288,V257,float64,0.736522,48,1.552443,0.380664,0.369300,candidata,0.026305,0.387241
274,V243,float64,0.736522,42,2.524546,0.164820,0.380319,candidata,0.027501,0.387195
277,V246,float64,0.736522,45,0.997453,0.351263,0.376941,candidata,0.022793,0.382640
289,V258,float64,0.736522,66,4.622362,0.256333,0.369131,candidata,0.025651,0.375544
232,V201,float64,0.723175,46,1.417571,0.352998,0.260591,candidata,0.024468,0.359115
218,V187,float64,0.723141,210,144.105923,0.013908,0.350491,candidata,0.016371,0.354583
231,V200,float64,0.723175,46,1.224610,0.337051,0.253861,candidata,0.023553,0.342939
221,V190,float64,0.723141,42,2.716580,0.148809,0.328603,candidata,0.026561,0.335243


<div class="section"><h2>3 · Correlación, ruido y PCA</h2></div>

Pearson, Spearman e información mutua miden formas distintas de asociación; ninguna demuestra causalidad. Constantes, ausencia extrema e IDs operativos se excluyen. PCA se ajusta por fold solo en numéricas elegibles: retener varianza no garantiza retener fraude.

In [4]:
pd.read_csv(ROOT/'artefactos/v2/validacion_walk_forward.csv').pivot(index='fold',columns='modelo',values='auc_pr')

modelo,CatBoost_nativo,LightGBM_PCA95,LightGBM_corr_pruned
fold,,,
F1,0.437974,0.417498,0.442998
F2,0.540141,0.534744,0.555317
F3,0.395844,0.395961,0.419750


<div class="section"><h2>4 · Walk-forward</h2></div>

Tres ventanas entrenan con pasado y evalúan futuro inmediato. Se comparan LightGBM depurado, CatBoost con categorías nativas y LightGBM+PCA95. El presupuesto CatBoost es acotado y se reporta honestamente.

In [5]:
pd.read_csv(ROOT/'artefactos/v2/ablacion_tamano_entrenamiento.csv')

,n_entrenamiento,auc_pr_validacion,roc_auc_validacion,mejor_iteracion
0,180000,0.466103,0.863015,500
1,300000,0.468186,0.862093,441
2,413378,0.466402,0.862672,417


<div class="section"><h2>5 · Tamaño de muestra</h2></div>

Se comparan 180k, 300k y todo el 70% disponible. Más datos no se declara mejor por definición: la deriva puede reducir utilidad de eventos antiguos.

In [6]:
pd.DataFrame({'LightGBM':R['modelo_tabular_v2']['benchmark_historico'],'Ensamble':R['ensamble_v2']['benchmark_historico']}).T[['auc_pr','precision','recall','f1','costo_q','alertas_por_100k']]

,auc_pr,precision,recall,f1,costo_q,alertas_por_100k
LightGBM,0.453579,0.148822,0.702887,0.245636,6078120.0,16438.062338
Ensamble,0.437905,0.141216,0.701914,0.235128,6228600.0,17299.420869


<div class="section"><h2>6 · Calibración y ensamble</h2></div>

El ensamble logístico combina score tabular y agregados causales. Reserva el final de validación para calibrar y escoger umbral. No se llama A+B-GRU porque no contiene predicciones OOF de una GRU V2.

In [7]:
pd.DataFrame(R['metricas_top_k'])

,tasa_revision,k,precision_at_k,recall_at_k
0,0.001,88,1.000000,0.028544
1,0.005,442,0.925339,0.132663
2,0.010,885,0.847458,0.243270
3,0.020,1771,0.632411,0.363283


<div class="section"><h2>7 · Resultados y capacidad</h2></div>

AUC-PR mide ranking; recall, fraude recuperado; precisión, carga; costo, consecuencia del umbral. Precision@K y Recall@K conectan el modelo con límites de revisión.

In [8]:
pd.DataFrame(R['intervalo_auc_pr_benchmark'],index=['valor']).T

,valor
estimacion,0.453579
li95,0.411303
ls95,0.4918
metodo,"bootstrap por 24 bloques temporales, 300 réplicas"


<div class="section"><h2>8 · Conclusión</h2></div>

Los datos preceden a la arquitectura. Correlación quita redundancia pero no decide sola; PCA se acepta solo si gana fuera de tiempo. Para una V3: cohorte nueva, predicciones OOF tabular+GRU, claves proxy, ventanas 3/8/16, Adam/AdamW y focal loss antes de TCN o atención.

In [9]:
R['limitaciones']

['El benchmark final ya fue observado en V1 y se reporta como histórico reutilizado.',
 'Las claves de identidad son proxies y pueden mezclar o fragmentar personas.',
 'Correlación e información mutua son filtros descriptivos, no pruebas de causalidad.',
 'PCA se trata como ablación; solo se adopta si mejora de forma estable la validación temporal.',
 'El costo FN/FP es un supuesto académico y requiere validación operativa antes de producción.']

<div class="callout warn"><b>Uso responsable.</b> Prototipo académico para priorizar revisión humana. No debe bloquear transacciones ni atribuir culpabilidad. Requiere privacidad, explicabilidad, seguridad, sesgo, monitoreo y costos reales antes de producción.</div>

<div class="section"><h2>9 · Referencias APA 7 y declaración de IA</h2></div>

Bai, S., Kolter, J. Z., & Koltun, V. (2018). *An empirical evaluation of generic convolutional and recurrent networks for sequence modeling*. arXiv. https://doi.org/10.48550/arXiv.1803.01271

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems, 30*. https://papers.neurips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree

Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). CatBoost: Unbiased boosting with categorical features. *Advances in Neural Information Processing Systems, 31*. https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

**Declaración de IA.** Se utilizó asistencia para estructurar código, redacción, visualización y auditoría. Los autores ejecutaron el experimento, verificaron las cifras y asumen responsabilidad por interpretación, seguridad y defensa.